In [3]:
from rds import get_rds_connection, create_table, create_index
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
create_table(conn)
create_index(conn)
conn.close()

Creating search table...
Search table created.
Creating index...
Index created.


In [1]:
from rds import get_rds_connection, backfill
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
backfill(conn)
conn.commit()
conn.close()

Backfilling theorem_search_qwen...
Rows inserted: 0


In [ ]:
from rds import get_rds_connection
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
cur = conn.cursor()

print("Configuring session...")

cur.execute("SET maintenance_work_mem = '8GB';")
cur.execute("SET max_parallel_maintenance_workers = 12;")
cur.execute("SET max_parallel_workers = 16;")
cur.execute("SET synchronous_commit = OFF;")

print("Creating index...")
cur.execute(r"""
CREATE INDEX CONCURRENTLY theorem_embedding_gemma_hnsw
ON theorem_embedding_gemma
USING hnsw (embedding vector_cosine_ops)
WITH (
  m = 16,
  ef_construction = 128
);
""")

print("Success.")

cur.close()
conn.close()

Configuring session...
Creating index...


In [4]:
from rds import get_rds_connection
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
cur = conn.cursor()

print("Executing query...")
cur.execute(r"""
CREATE TABLE arxiv_sample_10000 AS
WITH eligible AS (
  SELECT
    t.theorem_id,
    t.body,
    ts.slogan_id,
    ts.slogan,
    p.primary_category,
    p.categories
  FROM theorem t
  JOIN paper p
    ON p.paper_id = t.paper_id
  JOIN theorem_slogan ts
    ON ts.theorem_id = t.theorem_id
  WHERE p.source = 'arXiv'
    AND cardinality(p.categories) = 1
    AND p.categories[1] = ANY (ARRAY[
      'math.AP','math.CO','math.AG','math.PR','math.NT',
      'math.DG','math.DS','math.FA','math.RT','math.GR'
    ])
    AND ts.prompt_id = 'body-only-v1'
),
sampled AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY primary_category
      ORDER BY md5(theorem_id::text)
    ) AS rn
  FROM eligible
)
SELECT
  s.theorem_id,
  s.primary_category,
  s.body,
  s.slogan,
  r.embedding AS raw_embedding,
  g.embedding AS slogan_embedding
FROM sampled s
JOIN raw_theorem_embedding_gemma r
  ON r.theorem_id = s.theorem_id
JOIN theorem_embedding_gemma g
  ON g.slogan_id = s.slogan_id
WHERE s.rn <= 1000;
""")

print("Success.")

cur.close()
conn.close()

Executing query...
Success.
